In [1]:
import pandas as pd
import numpy as np
import numpyro
from jax import numpy as jnp
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer
from jax import jit
import matplotlib.pyplot as plt
import diffrax as dfx

from summer3.epi import CompartmentalModelODE, CategoryData, strat_data_from_pandas, build_istate, dti_to_epoch

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, MAX_AGE, DATA_PATH, LATENT_STATES
from tb_macro.epi import get_base_model, add_natural_history, add_seeding, add_latency_flows, add_infection_flows
from tb_macro.inputs import (
    get_country_pop,
    get_single_age_pop_from_ungroups,
    get_group_popsizes,
    get_un_mortality,
    add_groups_to_single_pop,
    build_age_weight_lookup,
    get_fertility_data,
    calc_tsr_from_outcomes,
    calc_death_in_unsucc_outcomes,
    get_country_indicators,
)
from tb_macro.demography import add_replacement_deaths, add_ageing_flows, prepare_pop_data_for_entries, add_entry_births
from tb_macro.health_system import add_treatment_flows, add_detection
from tb_macro.calibration import make_log_likelihood
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [2]:
iso3 = "VNM"
start_time = 1800.0
end_time = 2051.0
young_end_age = 15
plot_start = 1980.0
james_work_gdrive = "/Users/jtrauer/Library/CloudStorage/GoogleDrive-james.trauer@monash.edu/"

In [3]:
# Data loading and processing
pop_data = get_country_pop(iso3)
single_age_pops = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age_pops)
mort_data = get_un_mortality(iso3)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
add_groups_to_single_pop(single_age_pops)
age_weights = build_age_weight_lookup(single_age_pops)
fert = get_fertility_data(iso3)
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)
raw_outcome_data = pd.read_csv(DATA_PATH / "who/who_outcomes_20260514T0437Z.csv")
outcome_data = raw_outcome_data[raw_outcome_data["iso3"] == iso3]
tsr = calc_tsr_from_outcomes(outcome_data)
death_in_unsucc = calc_death_in_unsucc_outcomes(outcome_data)
who_indicators = get_country_indicators(iso3)
who_mort = who_indicators["e_mort_tbhiv_num"] + who_indicators["e_mort_exc_tbhiv_num"]

In [4]:
# Model construction
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(start_time, end_time)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, start_time, sum(start_apops))

add_infection_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat, age_weights, group_popsize, fert_padded, young_end_age, start_time)
add_natural_history(epi_model, disease_state, age_strat, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state, start_time)
add_detection(epi_model, disease_state, clin_strat, start_time)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, start_time)
add_entry_births(epi_model, disease_state, age_strat, start_time, entry_rates, entry_times)
add_treatment_flows(death_rates, start_time, epi_model, disease_state, age_strat, infect_strat, clin_strat, tsr, death_in_unsucc)
add_latency_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat)

# Initialisation
init_apops_series = pd.Series(index=[str(a) for a in AGE_STRATA], data=np.array(start_apops))
init_apops = strat_data_from_pandas(init_apops_series, age_strat)
init_dpops = [0.0] * len(ALL_COMPARTMENTS)
init_dpops[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_dpops)))]
epi_model.set_initial_population(init_apops, pop_splits)
epi_model.computed_values.append("dynamic_mm")

In [5]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [6]:
base_params = {
    "contact_rate": 7.0,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 0.3324848208129065,
    "rel_sus_cleared": 0.7101012998861186,
    "containment_age0": 4.4,
    "containment_age5": 4.4,
    "containment_age15": 2.0,
    "clearance_rate": 0.05643858771640714,
    "breakdown_rate": 0.5678500778834155,
    "progression_age0": 2.4,
    "progression_age5": 2.0,
    "progression_age15": 0.1,
    "progression_prop_infectious": 0.5,
    "increase_infect": 2.799432998282645,
    "decrease_infect": 1.0,
    "clinical_development": 2.280456422265371,
    "clinical_regression": 1.0,
    "self_recovery": 0.4,
    "seed_peak_time": 1830.0,
    "seed_peak_rate": 0.01,
    "seed_duration": 10.0,
    "detect_time_0": 1950.0,
    "detect_time_1": 1990.0,
    "detect_val_1": 0.2,
    "detect_time_2": 2010.0,
    "detect_val_2": 0.6,
    "detect_gap_reduction": 0.0,
    "a_spread": 9.832801834382463,
    "bg_mixing": 0.029353135750061505,
    "pc_strength": 0.9853729910300592,
    "young_suscept": 0.5,
    "rx_duration": 0.5,
    "rel_infectiousness_lowinf": 0.4,
    "rel_infectiousness_subclin": 0.5,
    "tb_mort_lowinf": 0.1,
    "tb_mort_inf": 0.1,
}
runner, istate = get_runner(epi_model)
solver_kwargs = {"max_steps": 4000}

In [7]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
latent_date = LATENT_TARGET.index[0]
latent_target_val = LATENT_TARGET.iloc[0] / 1e2

calib_params = ["contact_rate", "detect_val_2"]


def vector_to_params(calib_params, x):
    return dict(zip(calib_params, x))


def params_to_vector(calib_params, params):
    return np.array([params[p] for p in calib_params])


log_like = make_log_likelihood(epi_model, disease_state, solver_kwargs, latent_date, latent_target_val, NOTIF_TARGET, who_mort)

In [9]:
# Calibration
priors = {
    "contact_rate": dist.Uniform(5.0, 11.0),
    "detect_val_2": dist.Uniform(0.4, 0.7),
}


def model():
    params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
    ll = log_like(params)
    numpyro.factor("ll", ll)


# kernel = infer.SA(model)  # , adapt_state_size=4)  # infer.NUTS(model, max_tree_depth=5)
kernel = infer.NUTS(model, max_tree_depth=5, init_strategy=infer.init_to_median())
mcmc = infer.MCMC(kernel, num_warmup=100, num_samples=100, num_chains=4)
mcmc.run(PRNGKey(2))

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

W0716 21:56:24.425038 2208141 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.
